**CI twin of `ch17-kmeans.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

df = load_csv("penguins").dropna(subset=["flipper_length_mm",
                                         "bill_length_mm"])
Xs = StandardScaler().fit_transform(
    df[["flipper_length_mm", "bill_length_mm"]])

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.scatter(Xs[:, 0], Xs[:, 1], s=10, color="grey", alpha=0.5)
ax.set_xlabel("flipper length (scaled)")
ax.set_ylabel("bill length (scaled)")
plt.show()

In [ ]:
centres = Xs[[0, 170, 340]].copy()     # three arbitrary starting cards

def assign(points, centres):
    dists = ((points[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
    return dists.argmin(axis=1)        # index of each point's nearest centre

for it in range(1, 30):
    labels = assign(Xs, centres)
    new_centres = np.array([Xs[labels == j].mean(axis=0) for j in range(3)])
    move = float(np.abs(new_centres - centres).max())
    print(f"round {it}: pile sizes {np.bincount(labels).tolist()}, "
          f"largest centre move {move:.4f}")
    if move == 0:
        print(f"\nconverged — nothing moved in round {it}")
        break
    centres = new_centres

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xs)
print(f"pile sizes: {np.bincount(km.labels_).tolist()}")
print(f"inertia:    {km.inertia_:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.scatter(Xs[:, 0], Xs[:, 1], c=km.labels_, cmap="viridis", s=10, alpha=0.6)
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
           marker="x", s=120, color="red", label="centres")
ax.set_xlabel("flipper length (scaled)")
ax.set_ylabel("bill length (scaled)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
inertias = []
for k in range(1, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    inertias.append(m.inertia_)
    print(f"k={k}: inertia {m.inertia_:.1f}")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(range(1, 9), inertias, marker="o")
ax.set_xlabel("k (number of piles)")
ax.set_ylabel("inertia")
ax.annotate("the elbow", (3.1, 190))
plt.show()

In [ ]:
import pandas as pd

reveal = pd.crosstab(km.labels_, df["species"])
print(reveal)

agree = reveal.max(axis=1).sum()
print(f"\nbest-match agreement: {agree} of {len(df)} birds "
      f"({agree / len(df):.1%})")

In [ ]:
from lib.data import load_csv
from sklearn.preprocessing import StandardScaler
import numpy as np

df = load_csv("penguins").dropna(subset=["flipper_length_mm",
                                         "bill_length_mm"])
Xs = StandardScaler().fit_transform(
    df[["flipper_length_mm", "bill_length_mm"]])
centres = Xs[[0, 170, 340]].copy()

dists = ((Xs[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
labels = dists.argmin(axis=1)

run_tests([
    ("every card dealt", len(labels), 342),
    ("first-round pile sizes", np.bincount(labels).tolist(), [162, 98, 82]),
])

In [ ]:
def assign_to_centres(points, centres):
    def sq_dist(p, c):
        return sum((a - b) ** 2 for a, b in zip(p, c))
    return [min(range(len(centres)), key=lambda j: sq_dist(p, centres[j]))
            for p in points]

def update_centres(points, labels, k):
    out = []
    for j in range(k):
        pile = [p for p, lab in zip(points, labels) if lab == j]
        out.append(tuple(round(sum(coord) / len(pile), 4)
                         for coord in zip(*pile)))
    return out

pts = [(0.0, 0.0), (1.0, 0.0), (0.0, 1.0),
       (10.0, 10.0), (11.0, 10.0), (10.0, 11.0)]
cs = [(0.0, 0.0), (10.0, 10.0)]

run_tests([
    ("two clear piles", assign_to_centres(pts, cs), [0, 0, 0, 1, 1, 1]),
    ("centres move to pile middles",
     update_centres(pts, [0, 0, 0, 1, 1, 1], 2),
     [(0.3333, 0.3333), (10.3333, 10.3333)]),
    ("a second round changes nothing",
     assign_to_centres(pts, [(0.3333, 0.3333), (10.3333, 10.3333)]),
     [0, 0, 0, 1, 1, 1]),
])